In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, precision_score, recall_score
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import torch.optim as optim

# Implementation with TAAF switches

In [ ]:
# Define the CNN model with corrected channels
class TAAF(nn.Module):
    def __init__(self, in_features):
        super(TAAF, self).__init__()
        # Initialize learnable parameters for scaling and shifting
        self.alpha = nn.Parameter(torch.ones(1))  # Scale parameter (learnable)
        self.beta = nn.Parameter(torch.zeros(1))  # Shift parameter (learnable)

    def forward(self, x):
        # TAAF function can be some form of scaled, shifted non-linearity
        # For example, a simple TAAF could be a scaled and shifted ReLU
        return self.alpha * F.relu(x + self.beta)

class CNNModel(nn.Module):
    def __init__(self, num_of_classes=10): # Change 10 to the number of classes of the dataset
        super(CNNModel, self).__init__()

        self.conv1 = nn.Conv2d(1, 16, kernel_size=5, stride=1, padding=2, bias=False)  # 1 input channel
        self.bn1 = nn.BatchNorm2d(16)

        self.dwconv2 = nn.Conv2d(16, 16, kernel_size=3, padding=1, groups=16, bias=False)
        self.bn2 = nn.BatchNorm2d(16)

        self.dwconv3 = nn.Conv2d(16, 16, kernel_size=3, padding=1, groups=16, bias=False)
        self.bn3 = nn.BatchNorm2d(16)

        self.conv4 = nn.Conv2d(16, 32, kernel_size=3, padding=1, bias=False)  # 16 -> 32
        self.bn4 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.drop1 = nn.Dropout(0.15)

        self.dwconv5 = nn.Conv2d(32, 32, kernel_size=3, padding=1, groups=32, bias=False)
        self.bn5 = nn.BatchNorm2d(32)

        self.dwconv6 = nn.Conv2d(32, 32, kernel_size=3, padding=1, groups=32, bias=False)
        self.bn6 = nn.BatchNorm2d(32)

        self.conv7 = nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False)  # 32 -> 64
        self.bn7 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.drop2 = nn.Dropout(0.2)

        self.dwconv8 = nn.Conv2d(64, 64, kernel_size=3, padding=1, groups=64, bias=False)
        self.bn8 = nn.BatchNorm2d(64)

        self.dwconv9 = nn.Conv2d(64, 64, kernel_size=3, padding=1, groups=64, bias=False)
        self.bn9 = nn.BatchNorm2d(64)

        self.conv10 = nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False)
        self.bn10 = nn.BatchNorm2d(64)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.drop3 = nn.Dropout(0.2)

        self.dwconv11 = nn.Conv2d(64, 64, kernel_size=3, padding=1, groups=64, bias=False)
        self.bn11 = nn.BatchNorm2d(64)

        self.dwconv12 = nn.Conv2d(64, 64, kernel_size=3, padding=1, groups=64, bias=False)
        self.bn12 = nn.BatchNorm2d(64)

        self.conv13 = nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False)
        self.bn13 = nn.BatchNorm2d(64)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.drop4 = nn.Dropout(0.2)

        self.dwconv14 = nn.Conv2d(64, 64, kernel_size=3, padding=1, groups=64, bias=False)
        self.bn14 = nn.BatchNorm2d(64)

        self.dwconv15 = nn.Conv2d(64, 64, kernel_size=3, padding=1, groups=64, bias=False)
        self.bn15 = nn.BatchNorm2d(64)

        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.drop5 = nn.Dropout(0.25)
        self.fc = nn.Linear(64, num_of_classes)

        self.taaf = TAAF(1024)

    def forward(self, x):
        x = self.taaf(self.bn1(self.conv1(x)))
        x = self.taaf(self.bn2(self.dwconv2(x)))
        x = self.taaf(self.bn3(self.dwconv3(x)))

        x = self.taaf(self.bn4(self.conv4(x)))
        x = self.pool1(x)
        x = self.drop1(x)

        x = self.taaf(self.bn5(self.dwconv5(x)))
        x = self.taaf(self.bn6(self.dwconv6(x)))

        x = self.taaf(self.bn7(self.conv7(x)))
        x = self.pool2(x)
        x = self.drop2(x)

        x = self.taaf(self.bn8(self.dwconv8(x)))
        x = self.taaf(self.bn9(self.dwconv9(x)))

        x = self.taaf(self.bn10(self.conv10(x)))
        x = self.pool3(x)
        x = self.drop3(x)

        x = self.taaf(self.bn11(self.dwconv11(x)))
        x = self.taaf(self.bn12(self.dwconv12(x)))

        x = self.taaf(self.bn13(self.conv13(x)))
        x = self.pool4(x)
        x = self.drop4(x)

        x = self.taaf(self.bn14(self.dwconv14(x)))
        x = self.taaf(self.bn15(self.dwconv15(x)))

        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        x = self.drop5(x)
        x = self.fc(x)
        return x

# Initialize model
model = CNNModel(num_of_classes=num_of_classes)

# Device management (uses GPU if possible)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Optimizer
optimizer = optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)

# Learning Rate Scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5, min_lr=2.5e-5)

# Early Stopping
class EarlyStopping:
    def __init__(self, patience=10, restore_best_weights=True):
        self.patience = patience
        self.restore_best_weights = restore_best_weights
        self.best_loss = float('inf')
        self.counter = 0
        self.best_model_weights = None

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.best_model_weights = model.state_dict() if self.restore_best_weights else None
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                print("Early stopping triggered!")
                if self.restore_best_weights and self.best_model_weights is not None:
                    model.load_state_dict(self.best_model_weights)
                return True
        return False

early_stopping = EarlyStopping(patience=10, restore_best_weights=True)

# Implementation without TAAF switches

In [ ]:
# Define the CNN model with corrected channels
class CNNModel(nn.Module):
    def __init__(self, num_of_classes=62):
        super(CNNModel, self).__init__()

        self.conv1 = nn.Conv2d(1, 16, kernel_size=5, stride=1, padding=2, bias=False)  # 1 input channel
        self.bn1 = nn.BatchNorm2d(16)

        self.dwconv2 = nn.Conv2d(16, 16, kernel_size=3, padding=1, groups=16, bias=False)
        self.bn2 = nn.BatchNorm2d(16)

        self.dwconv3 = nn.Conv2d(16, 16, kernel_size=3, padding=1, groups=16, bias=False)
        self.bn3 = nn.BatchNorm2d(16)

        self.conv4 = nn.Conv2d(16, 32, kernel_size=3, padding=1, bias=False)  # 16 -> 32
        self.bn4 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.drop1 = nn.Dropout(0.15)

        self.dwconv5 = nn.Conv2d(32, 32, kernel_size=3, padding=1, groups=32, bias=False)
        self.bn5 = nn.BatchNorm2d(32)

        self.dwconv6 = nn.Conv2d(32, 32, kernel_size=3, padding=1, groups=32, bias=False)
        self.bn6 = nn.BatchNorm2d(32)

        self.conv7 = nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False)  # 32 -> 64
        self.bn7 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.drop2 = nn.Dropout(0.2)

        self.dwconv8 = nn.Conv2d(64, 64, kernel_size=3, padding=1, groups=64, bias=False)
        self.bn8 = nn.BatchNorm2d(64)

        self.dwconv9 = nn.Conv2d(64, 64, kernel_size=3, padding=1, groups=64, bias=False)
        self.bn9 = nn.BatchNorm2d(64)

        self.conv10 = nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False)
        self.bn10 = nn.BatchNorm2d(64)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.drop3 = nn.Dropout(0.2)

        self.dwconv11 = nn.Conv2d(64, 64, kernel_size=3, padding=1, groups=64, bias=False)
        self.bn11 = nn.BatchNorm2d(64)

        self.dwconv12 = nn.Conv2d(64, 64, kernel_size=3, padding=1, groups=64, bias=False)
        self.bn12 = nn.BatchNorm2d(64)

        self.conv13 = nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False)
        self.bn13 = nn.BatchNorm2d(64)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.drop4 = nn.Dropout(0.2)

        self.dwconv14 = nn.Conv2d(64, 64, kernel_size=3, padding=1, groups=64, bias=False)
        self.bn14 = nn.BatchNorm2d(64)

        self.dwconv15 = nn.Conv2d(64, 64, kernel_size=3, padding=1, groups=64, bias=False)
        self.bn15 = nn.BatchNorm2d(64)

        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.drop5 = nn.Dropout(0.25)
        self.fc = nn.Linear(64, num_of_classes)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.dwconv2(x)))
        x = F.relu(self.bn3(self.dwconv3(x)))

        x = F.relu(self.bn4(self.conv4(x)))
        x = self.pool1(x)
        x = self.drop1(x)

        x = F.relu(self.bn5(self.dwconv5(x)))
        x = F.relu(self.bn6(self.dwconv6(x)))

        x = F.relu(self.bn7(self.conv7(x)))
        x = self.pool2(x)
        x = self.drop2(x)

        x = F.relu(self.bn8(self.dwconv8(x)))
        x = F.relu(self.bn9(self.dwconv9(x)))

        x = F.relu(self.bn10(self.conv10(x)))
        x = self.pool3(x)
        x = self.drop3(x)

        x = F.relu(self.bn11(self.dwconv11(x)))
        x = F.relu(self.bn12(self.dwconv12(x)))

        x = F.relu(self.bn13(self.conv13(x)))
        x = self.pool4(x)
        x = self.drop4(x)

        x = F.relu(self.bn14(self.dwconv14(x)))
        x = F.relu(self.bn15(self.dwconv15(x)))

        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        x = self.drop5(x)
        x = self.fc(x)
        return x

# Initialize model
model = CNNModel(num_of_classes=num_of_classes)

# Device management
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Optimizer
optimizer = optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)

# Learning Rate Scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5, min_lr=2.5e-5)

# Early Stopping
class EarlyStopping:
    def __init__(self, patience=10, restore_best_weights=True):
        self.patience = patience
        self.restore_best_weights = restore_best_weights
        self.best_loss = float('inf')
        self.counter = 0
        self.best_model_weights = None

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.best_model_weights = model.state_dict() if self.restore_best_weights else None
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                print("Early stopping triggered!")
                if self.restore_best_weights and self.best_model_weights is not None:
                    model.load_state_dict(self.best_model_weights)
                return True
        return False

early_stopping = EarlyStopping(patience=10, restore_best_weights=True)